# Hay micro-neuron: dataset sinaptico a quattro compartimenti

Teacher ridotto **soma + basale + tronco apicale + tuft**, senza corrente iniettata. Gli input sono esclusivamente spike presinaptici binari su sinapsi fisse e gli output contengono tutti i 61 stati fisici.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = 'https://github.com/Zagred47/LearningSingleCompartiment.git'
ROOT = Path('/kaggle/working/LearningSingleCompartiment')
if (ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif ROOT.exists():
    print('Directory esistente ma non è un clone Git; uso i file presenti:', ROOT)
else:
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Project root:', ROOT)

In [ ]:
import json, time
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hay_single_compartment import (
    MICRO_REGIME_NAMES, MICRO_STATE_NAMES, FourCompartmentHay,
    MicroDatasetConfig, build_micro_synapse_metadata,
    generate_micro_dataset, validate_micro_dataset,
)

CONFIG = MicroDatasetConfig()
WORKING_DATASET = Path('/kaggle/working/hay_micro_4c_v1.h5')
mounted = sorted(Path('/kaggle/input').glob('**/hay_micro_4c_v1.h5'))
DATASET_PATH = mounted[0] if mounted else WORKING_DATASET
WORKERS = min(4, os.cpu_count() or 1)
print('Dataset:', DATASET_PATH)
print('Workers:', WORKERS)
print('Traiettorie:', CONFIG.train_trajectories, CONFIG.validation_trajectories, CONFIG.test_trajectories)
print('Durata utile per traiettoria:', CONFIG.duration_ms, 'ms; warm-up:', CONFIG.warmup_ms, 'ms')
print('Stati:', len(MICRO_STATE_NAMES), 'Input sinaptici:', len(build_micro_synapse_metadata(CONFIG)))

In [ ]:
# Questa cella stampa avanzamento, tempo trascorso ed ETA dopo ogni traiettoria.
if DATASET_PATH.is_file() and DATASET_PATH != WORKING_DATASET:
    report = validate_micro_dataset(DATASET_PATH)
    if not report['valid']:
        raise ValueError(f'Dataset montato incompatibile: {report["issues"]}')
    report.update({'cache_hit': True, 'path': str(DATASET_PATH)})
else:
    started = time.perf_counter()
    report = generate_micro_dataset(
        WORKING_DATASET, CONFIG, workers=WORKERS, progress=True, reuse=True
    )
    DATASET_PATH = WORKING_DATASET
    print(f'Tempo totale: {(time.perf_counter() - started) / 60:.1f} min')
print(json.dumps(report, indent=2))

In [ ]:
with h5py.File(DATASET_PATH, 'r') as h5:
    state_names = json.loads(h5.attrs['state_names_json'])
    input_metadata = json.loads(h5.attrs['input_metadata_json'])
    current_names = json.loads(h5.attrs['current_names_json'])
    burnin_inputs = h5['train/burnin_inputs'][0]
    inputs = h5['train/inputs'][0]
    states = h5['train/states'][0]
    regimes = h5['train/regimes'][0]
    output_spikes = h5['train/spikes'][0]
    time_ms = h5['time_ms'][...]
print('Input soltanto binari:', np.isin(inputs, (0, 1)).all())
print('Burn-in input-only salvato:', burnin_inputs.shape)
print('Corrente iniettata:', False)
display(pd.DataFrame(input_metadata))

In [ ]:
window_ms = min(1000.0, CONFIG.duration_ms)
end = int(round(window_ms / CONFIG.dt_ms))
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True, constrained_layout=True)
event_t, event_s = np.nonzero(inputs[:end])
colors = np.asarray(['tab:red' if row['kind'] == 'excitatory' else 'tab:blue' for row in input_metadata])
axes[0].scatter(event_t * CONFIG.dt_ms, event_s, s=7, c=colors[event_s], alpha=.8)
axes[0].set_ylabel('synapse id'); axes[0].set_title('Input presinaptico binario')
for region in ('soma', 'basal', 'trunk', 'tuft'):
    axes[1].plot(time_ms[:end+1], states[:end+1, state_names.index(region + '.v_mV')], label=region, lw=1)
axes[1].set_ylabel('V (mV)'); axes[1].legend(ncol=4)
for region in ('soma', 'trunk', 'tuft'):
    axes[2].plot(time_ms[:end+1], 1000 * states[:end+1, state_names.index(region + '.ca_i_mM')], label=region, lw=1)
axes[2].set_ylabel('Ca_i (uM)'); axes[2].set_xlabel('time (ms)'); axes[2].legend(ncol=3)
plt.show()

In [ ]:
coverage = pd.DataFrame({
    'regime': MICRO_REGIME_NAMES,
    'samples': np.bincount(regimes, minlength=len(MICRO_REGIME_NAMES)),
})
coverage['fraction'] = coverage.samples / coverage.samples.sum()
display(coverage)
print('Spike presinaptici:', int(inputs.sum()))
print('Spike somatici:', int(output_spikes.sum()))
print('Range globale V:', float(states[:, [state_names.index(r + '.v_mV') for r in ('soma','basal','trunk','tuft')]].min()),
      float(states[:, [state_names.index(r + '.v_mV') for r in ('soma','basal','trunk','tuft')]].max()))

In [ ]:
# Stati lenti: la loro presenza nel file è parte del contratto, non un'opzione.
slow_names = [
    'soma.h_Nap_Et2', 'soma.h_K_Pst', 'soma.m_Ih',
    'basal.m_Ih', 'trunk.m_Ih', 'tuft.m_Ih',
    'trunk.m_Im', 'tuft.m_Im',
]
fig, ax = plt.subplots(figsize=(16, 5), constrained_layout=True)
for name in slow_names:
    ax.plot(time_ms, states[:, state_names.index(name)], label=name, lw=1)
ax.set(xlabel='time (ms)', ylabel='gate', title='Variabili lente conservate')
ax.legend(ncol=4)
plt.show()

## Contratto per il modello neurale

Il modello riceverà esclusivamente `inputs[t, synapse_id]`. `states` e `currents` sono target/diagnostica e non devono essere concatenati agli input. Il modello input-only verrà aggiunto come esperimento separato per non confondere la verifica del teacher con una nuova ipotesi architetturale.

In [ ]:
# Crea un archivio piccolo con dataset e manifest, scaricabile dal pannello Output di Kaggle.
from shutil import copy2, make_archive
EXPORT_DIR = Path('/kaggle/working/hay_micro_4c_export')
EXPORT_DIR.mkdir(exist_ok=True)
target = EXPORT_DIR / DATASET_PATH.name
if DATASET_PATH.resolve() != target.resolve():
    copy2(DATASET_PATH, target)
manifest = DATASET_PATH.with_suffix('.manifest.json')
if manifest.exists():
    copy2(manifest, EXPORT_DIR / manifest.name)
zip_path = Path(make_archive('/kaggle/working/hay_micro_4c_v1_complete', 'zip', root_dir=EXPORT_DIR.parent, base_dir=EXPORT_DIR.name))
print('Archivio pronto:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')
print('Scaricalo da /kaggle/working oppure dalla sezione Output dopo Save Version.')